# Combined Post-Task Likert Scale Analysis

This notebook combines post-task Likert scale data from all three user studies (diversity, compare distributions, single distribution) and runs Wilcoxon signed-rank tests between graph vs list interfaces for each question.

**Output**: A table with questions on the y-axis (grouped by section), columns grouped by user study, showing effect size (r) and p-value. Effect size cells are colored with a divergent scale (-1 to 1) where positive = graph favored, negative = list favored.

## 1. Setup and Question Mapping

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pingouin as pg
import re
from pathlib import Path
from IPython.display import display

# Resolve paths: notebook may run from combined_post_survey or project root
BASE = Path('.').resolve()
if (BASE / 'diversity').exists():
    pass  # cwd is user_study/process_user_study_results
elif (BASE / 'user_study/process_user_study_results' / 'diversity').exists():
    BASE = BASE / 'user_study/process_user_study_results'
else:
    BASE = BASE.parent  # e.g. cwd is combined_post_survey

# Question categorization (exclude "This study is about elephant behavior")
SECTIONS = {
    "Understanding model behavior": [
        '\"Using this interface, I understood how diverse (ie, how narrow or broad) the output space was for a given prompt\"',
        "I understood what a typical (or 'average') output looked like for a given prompt. ",
        "I had a good sense of what rare or unusual outputs looked like for a given prompt.",
        "I felt I had seen enough of the model's behavior for a prompt to make a good decision.",
        "I could see if there were recurring patterns in the outputs, and if so, what types",
    ],
    "Workload and Effort (NASA TLX)": [
        "Using this interface required a lot of mental effort.",
        "I had to work hard to complete the task using this interface.",
        "I felt frustrated while using this interface.",
        "I felt rushed while using this interface.",
    ],
    "Usability and satisfaction": [
        "I found the interface easy to use.",
        "I felt confident in the decisions I made using this interface.",
        "I would want to use this interface for similar tasks in my own work.",
    ],
}

def find_column(df, partial_name):
    """Find column that contains the partial name (handles encoding/quote variations)."""
    def norm(s):
        s = str(s).replace('"', '').replace('"', '').replace('"', '').replace('\\"', '').strip()
        return s.lower()
    partial_clean = norm(partial_name)
    for c in df.columns:
        if partial_clean in norm(c):
            return c
    return None

def wilcoxon_effect_size(x, y):
    """Run Wilcoxon signed-rank test (paired) via pingouin, return (pvalue, r)."""
    mask = ~(np.isnan(x) | np.isnan(y))
    x, y = np.asarray(x)[mask].astype(float), np.asarray(y)[mask].astype(float)
    if len(x) < 3:
        return 1.0, np.nan
    try:
        res = pg.wilcoxon(x, y, alternative='two-sided')
        pcol = 'p_val' if 'p_val' in res.columns else 'p-val'
        p = float(res[pcol].iloc[0])
        r = float(res['RBC'].iloc[0]) if 'RBC' in res.columns else np.nan
        return p, r
    except Exception:
        return 1.0, np.nan


## 2. Load Diversity Study Data

Diversity and compare_distributions: each row has two blocks of 12 Likert columns (cols 1-12 = first interface, 13-24 = second). Metadata `vis_type` indicates order: `graph,raw_outputs` means graph first, `raw_outputs,graph` means list first.

In [ ]:
def load_diversity_or_compare(path, study_name):
    """Load diversity or compare_distributions CSV. Returns list of result dicts."""
    df = pd.read_csv(path)
    metadata_col = [c for c in df.columns if 'Metadata' in c or 'metadata' in c.lower()]
    metadata_col = metadata_col[0] if metadata_col else None

    def get_vis_type(meta):
        if pd.isna(meta):
            return None
        m = re.search(r'vis_type=([^&]+)', str(meta))
        return m.group(1).strip() if m else None

    def _graph_first(m):
        v = get_vis_type(m)
        return v is not None and v.startswith('graph')
    df['_graph_first'] = df[metadata_col].apply(_graph_first) if metadata_col else True

    # Block1 = cols 1-12, block2 = 13,14,15,16,18,19,20,21,22,23,24,25 (skip 17=elephant)
    block1_idx = list(range(1, 13))
    block2_idx = [13, 14, 15, 16, 18, 19, 20, 21, 22, 23, 24, 25]
    headers = list(df.columns)

    q_order = []
    for section, questions in SECTIONS.items():
        for q in questions:
            q_order.append((section, q))

    results = []
    for i, (section, q) in enumerate(q_order):
        if i >= 12:
            break
        col1, col2 = headers[block1_idx[i]], headers[block2_idx[i]]
        v1 = pd.to_numeric(df[col1], errors='coerce')
        v2 = pd.to_numeric(df[col2], errors='coerce')
        # Per-row: graph_first means block1=graph, block2=list; else block1=list, block2=graph
        g_vals = np.where(df['_graph_first'], v1, v2)
        l_vals = np.where(df['_graph_first'], v2, v1)
        mask = ((v1 >= 1) & (v1 <= 7) & (v2 >= 1) & (v2 <= 7))
        g_paired = g_vals[mask]
        l_paired = l_vals[mask]

        p, r = wilcoxon_effect_size(g_paired, l_paired)
        r_display = -r if "Workload" in section else r
        results.append({
            'section': section,
            'question': q[:55] + ('...' if len(q) > 55 else ''),
            'question_full': q,
            'study': study_name,
            'p': p,
            'r': r,
            'r_display': r_display,
            'n': int(mask.sum()),
        })
    return results

# Run for diversity
div_path = BASE / 'diversity' / 'user_study_logging - diversity.csv'
div_results = load_diversity_or_compare(div_path, 'Diversity')
print(f"Diversity: {len(div_results)} question results")

In [ ]:
# Run for compare distributions
comp_path = BASE / 'compare_distributions' / 'user_study_logging - comparing comparisons.csv'
comp_results = load_diversity_or_compare(comp_path, 'Compare distributions')
print(f"Compare distributions: {len(comp_results)} question results")

## 3. Load Single Distribution Data

Single distribution: each participant filled out the survey twice (once per interface). Interface is indicated by `vis_type` in metadata (`graph` vs `raw_outputs`), not "Which interface were you using?". We pivot by prolific_pid to get paired graph vs list.

In [ ]:
def load_single_distribution(path):
    df = pd.read_csv(path)
    metadata_col = [c for c in df.columns if 'Metadata' in c or 'metadata' in c.lower()]
    metadata_col = metadata_col[0] if metadata_col else None

    def parse_pid(meta):
        if pd.isna(meta) or not isinstance(meta, str):
            return None
        m = re.search(r'prolific_pid=([^&]+)', meta)
        return m.group(1).strip() if m else None
    
    df['prolific_pid'] = df[metadata_col].apply(parse_pid) if metadata_col else None
    # Interface from vis_type in metadata (graph vs raw_outputs), not "Which interface were you using?"
    def get_vis_type(meta):
        if pd.isna(meta):
            return None
        m = re.search(r'vis_type=([^&]+)', str(meta))
        return m.group(1).strip() if m else None
    def _interface_from_vis(v):
        if not v:
            return None
        v = v.lower()
        if 'graph' in v:
            return 'graph'
        if 'raw' in v:
            return 'list'
        return None
    df['interface'] = df[metadata_col].apply(get_vis_type).apply(_interface_from_vis) if metadata_col else None
    
    # Single dist: Likert cols 32,33,34,35,37,38,39,40,41,42,43,44 (skip 36=elephant)
    single_dist_col_idx = [32, 33, 34, 35, 37, 38, 39, 40, 41, 42, 43, 44]
    q_order = [(s, q) for s, qs in SECTIONS.items() for q in qs]
    headers = list(df.columns)

    results = []
    for i, (section, q) in enumerate(q_order):
        if i >= len(single_dist_col_idx):
            break
        col = headers[single_dist_col_idx[i]] if single_dist_col_idx[i] < len(headers) else None
        if col is None:
            col = find_column(df, q.replace('\\"', '"'))
        if col is None:
            continue

        sub = df[df['interface'].notna() & df['prolific_pid'].notna()].copy()
        pivot = sub.pivot_table(index='prolific_pid', columns='interface', values=col, aggfunc='first')
        if 'graph' not in pivot.columns or 'list' not in pivot.columns:
            continue
        g_vals = pd.to_numeric(pivot['graph'], errors='coerce')
        l_vals = pd.to_numeric(pivot['list'], errors='coerce')
        mask = ((g_vals >= 1) & (g_vals <= 7) & (l_vals >= 1) & (l_vals <= 7))
        g_paired = g_vals[mask].values
        l_paired = l_vals[mask].values

        p, r = wilcoxon_effect_size(g_paired, l_paired)
        r_display = -r if "Workload" in section else r
        results.append({
            'section': section,
            'question': q[:55] + ('...' if len(q) > 55 else ''),
            'question_full': q,
            'study': 'Single distribution',
            'p': p,
            'r': r,
            'r_display': r_display,
            'n': int(mask.sum()),
        })
    return results

single_path = BASE / 'single_distribution' / 'user_study_logging - Single Distribution.csv'
single_results = load_single_distribution(single_path)
print(f"Single distribution: {len(single_results)} question results")

## 4. Build Combined Results Table and Heatmap

In [ ]:
all_results = div_results + comp_results + single_results
results_df = pd.DataFrame(all_results)

# Short labels for questions - match by substring since quote encoding may vary
def get_short_label(q_full):
    key_map = [
        ("understood how diverse", "Understood diversity of output space"),
        ("typical", "Understood typical output"),
        ("rare or unusual", "Understood rare/unusual outputs"),
        ("seen enough", "Seen enough for good decision"),
        ("recurring patterns", "Could see recurring patterns"),
        ("mental effort", "Required mental effort"),
        ("work hard", "Had to work hard"),
        ("frustrated", "Felt frustrated"),
        ("rushed", "Felt rushed"),
        ("easy to use", "Interface easy to use"),
        ("confident in the decisions", "Felt confident in decisions"),
        ("would want to use", "Would use for own work"),
    ]
    q_lower = str(q_full).lower().replace('"', '').replace('"', '')
    for key, label in key_map:
        if key in q_lower:
            return label
    return q_full[:50] if len(str(q_full)) > 50 else str(q_full)

# Build pivot: rows = (section, question), cols = study with (r_display, p)
studies = ['Diversity', 'Compare distributions', 'Single distribution']

# Row order: follow SECTIONS
section_order = list(SECTIONS.keys())
row_keys = []
for sec in section_order:
    for q in SECTIONS[sec]:
        q_short = q[:55] + ('...' if len(q) > 55 else '')
        row_keys.append((sec, q_short, q))

# Build matrix: rows x (studies * 2) for r and p
n_rows = len(row_keys)
n_studies = len(studies)
r_matrix = np.full((n_rows, n_studies), np.nan)
p_matrix = np.full((n_rows, n_studies), np.nan)

for i, (sec, q_short, q_full) in enumerate(row_keys):
    for j, study in enumerate(studies):
        match = results_df[(results_df['section'] == sec) & (results_df['study'] == study)]
        # Match by question (full or short)
        for _, row in match.iterrows():
            if row['question_full'] == q_full or row['question'] == q_short:
                r_matrix[i, j] = row['r_display']
                p_matrix[i, j] = row['p']
                break

print("Results matrix shape:", r_matrix.shape)
print("Row labels (section, question):")
for i, (sec, q, _) in enumerate(row_keys):
    print(f"  {i}: {sec} | {q[:50]}...")

In [ ]:
# Create figure: table with effect size heatmap and p-values
fig, ax = plt.subplots(figsize=(10, max(6, n_rows * 0.35)))

# Effect size heatmap (divergent -1 to 1)
vmin, vmax = -1, 1
im = ax.imshow(r_matrix, aspect='auto', cmap='RdBu_r', vmin=vmin, vmax=vmax)

# Labels
ax.set_xticks(np.arange(n_studies))
ax.set_xticklabels(studies, fontsize=10)
ax.set_yticks(np.arange(n_rows))
row_labels = [get_short_label(q_full) for _, _, q_full in row_keys]
ax.set_yticklabels(row_labels, fontsize=9)

# Section dividers (horizontal lines between sections)
sec_starts = {}
for i, (sec, _, _) in enumerate(row_keys):
    if sec not in sec_starts:
        sec_starts[sec] = i
for sec, start in sec_starts.items():
    if start > 0:
        ax.axhline(start - 0.5, color='black', linewidth=1)
ax.axhline(n_rows - 0.5, color='black', linewidth=1)

# Add text annotations: r and p in each cell
for i in range(n_rows):
    for j in range(n_studies):
        r_val = r_matrix[i, j]
        p_val = p_matrix[i, j]
        if np.isnan(r_val):
            text = '—'
        else:
            r_str = f"{r_val:.2f}"
            p_str = f"{p_val:.3f}" if p_val >= 0.001 else '<0.001'
            text = f"r={r_str}\np={p_str}"
        color = 'white' if not np.isnan(r_val) and abs(r_val) > 0.5 else 'black'
        ax.text(j, i, text, ha='center', va='center', fontsize=8, color=color)

plt.colorbar(im, ax=ax, label='Effect size (r): positive=graph favored, negative=list favored')
ax.set_title('Wilcoxon Signed-Rank Test: Graph vs List by Study')
plt.tight_layout()

# Save for paper (latex/figures/)
fig_path = BASE.parent / 'latex' / 'figures' / 'likert_post_survey.png'
if not fig_path.parent.exists():
    fig_path = Path('likert_post_survey.png')
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f"Saved figure to {fig_path}")
plt.show()

In [ ]:
# Also output as a proper table (DataFrame) for export
table_rows = []
for i, (sec, q_short, _) in enumerate(row_keys):
    row = {'Section': sec, 'Question': q_short}
    for j, study in enumerate(studies):
        r_val = r_matrix[i, j]
        p_val = p_matrix[i, j]
        row[f'{study} (r)'] = f"{r_val:.2f}" if not np.isnan(r_val) else '—'
        row[f'{study} (p)'] = f"{p_val:.3f}" if not np.isnan(p_val) and p_val >= 0.001 else ('<0.001' if not np.isnan(p_val) else '—')
    table_rows.append(row)

table_df = pd.DataFrame(table_rows)
display(table_df)

## 5. Figure for Paper

The heatmap above is saved to `latex/figures/likert_post_survey.png` for inclusion in the paper. Uses purple/green color scale (list/graph) from other notebooks.

In [ ]:
# Colors from other notebooks: list=#8b5a5a (purple/maroon), graph=#7ab89a (green)
LIST_COLOR = (139/255, 90/255, 90/255)   # #8b5a5a
GRAPH_COLOR = (122/255, 184/255, 154/255)  # #7ab89a

def r_to_latex_color(r_val):
    """Map r in [-1,1] to RGB between list (purple) and graph (green), through white at 0."""
    if np.isnan(r_val):
        return (0.95, 0.95, 0.95)  # light gray for missing
    r_val = np.clip(float(r_val), -1, 1)
    if r_val < 0:
        t = (r_val + 1) / 2  # 0 when r=-1, 0.5 when r=0
        return tuple(LIST_COLOR[i] * (1 - 2*t) + 1.0 * (2*t) for i in range(3))
    else:
        t = r_val  # 0 when r=0, 1 when r=1
        return tuple(1.0 * (1 - t) + GRAPH_COLOR[i] * t for i in range(3))

def latex_escape(s):
    return s.replace('&', '\\&').replace('_', '\\_').replace('%', '\\%')

# Build LaTeX table (for copy/paste into paper)
lines = []
lines.append(r"\begin{figure*}[t]")
lines.append(r"  \centering")
lines.append(r"  \small")
lines.append(r"  \begin{tabular}{@{}p{4.8cm}ccc@{}}")
lines.append(r"  \toprule")
lines.append(r"  & " + " & ".join(studies) + r" \\")
lines.append(r"  \midrule")

prev_sec = None
for i, (sec, q_short, q_full) in enumerate(row_keys):
    if sec != prev_sec:
        if prev_sec is not None:
            lines.append(r"  \addlinespace[0.5em]")
        lines.append(r"  \multicolumn{4}{@{}l}{\textit{" + latex_escape(sec) + r"}} \\")
        prev_sec = sec

    q_label = latex_escape(get_short_label(q_full))
    cell_parts = []
    for j in range(n_studies):
        r_val = r_matrix[i, j]
        p_val = p_matrix[i, j]
        if np.isnan(r_val):
            cell_parts.append("---")
        else:
            r_str = f"{r_val:.2f}"
            p_str = f"{p_val:.3f}" if p_val >= 0.001 else "<0.001"
            cell_parts.append(f"r={r_str}, $p${'=' + p_str if p_str != '<0.001' else '<0.001'}")

    row_str = "  " + q_label
    for j in range(n_studies):
        r_val = r_matrix[i, j]
        rgb = r_to_latex_color(r_val)
        rgb_str = f"{rgb[0]:.3f},{rgb[1]:.3f},{rgb[2]:.3f}"
        row_str += f" & \\cellcolor[rgb]{{{rgb_str}}} {cell_parts[j]}"
    lines.append(row_str + r" \\")

lines.append(r"  \bottomrule")
lines.append(r"  \end{tabular}")
lines.append(r"  \caption{Post-task Likert scale: Wilcoxon signed-rank test (graph vs.\\ list) for each question across studies. Effect size $r$ (rank-biserial correlation) and $p$-value. Positive $r$ favors graph; negative $r$ favors list. Colors: purple (list favored) to green (graph favored).}")
lines.append(r"  \label{fig:likert_post_survey}")
lines.append(r"\end{figure*}")

print("\n".join(lines))
